# SDXL × DrawBench — Qualitative Results

Side-by-side comparison of both sampling strategies 

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
IMAGES_DIR = Path("sdxl_drawbench_images")
STANDARD_DIR = IMAGES_DIR / "sdxl_drawbench"
ZSAMPLING_DIR = IMAGES_DIR / "sdxl_drawbench_zsampling"

CSV_PATH = Path("sdxl_drawbench_eval_results.csv")

NUM_SAMPLES = 6
RNG_SEED = 42

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total evaluation pairs: {len(df)}")
df.head()

In [ ]:
def index_images(base_dir: Path) -> dict[str, Path]:
    """Build filename -> path index across all checkpoint subfolders."""
    return {p.name: p for p in base_dir.rglob("*.png")}

std_index = index_images(STANDARD_DIR)
zs_index = index_images(ZSAMPLING_DIR)

print(f"Standard images: {len(std_index)}")
print(f"Z-Sampling images: {len(zs_index)}")

In [ ]:
rng = np.random.default_rng(RNG_SEED)
sample_indices = rng.choice(len(df), size=NUM_SAMPLES, replace=False)
samples = df.iloc[sample_indices].reset_index(drop=True)

print("Selected prompts:")
for _, row in samples.iterrows():
    print(f"  [{row['prompt_idx']:03d}, seed={row['seed']}] {row['prompt']}")

In [ ]:
fig, axes = plt.subplots(2, NUM_SAMPLES, figsize=(3.5 * NUM_SAMPLES, 7.5))

for col, (_, row) in enumerate(samples.iterrows()):
    idx = int(row["prompt_idx"])
    seed = int(row["seed"])
    prompt = row["prompt"]

    std_fname = f"sdxl_prompt{idx:03d}_seed{seed}.png"
    zs_fname = f"sdxl_zsample_prompt{idx:03d}_seed{seed}.png"

    std_img = Image.open(std_index[std_fname])
    zs_img = Image.open(zs_index[zs_fname])

    axes[0, col].imshow(std_img)
    axes[1, col].imshow(zs_img)

    wrapped = "\n".join(textwrap.wrap(prompt, width=22))
    axes[0, col].set_title(wrapped, fontsize=9, pad=6)

    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([])
        ax.set_yticks([])

axes[0, 0].set_ylabel("Standard\nSampling", fontsize=11, rotation=0, labelpad=60, va="center")
axes[1, 0].set_ylabel("Z-Sampling\n(Ours)", fontsize=11, rotation=0, labelpad=60, va="center")

fig.suptitle("SDXL × DrawBench: Standard Sampling vs Z-Sampling", fontsize=14, y=0.98)
plt.tight_layout(rect=[0.05, 0, 1, 0.95])
plt.savefig("sdxl_drawbench_qualitative.png", dpi=150, bbox_inches="tight")
plt.show()